# H&M 전체 2년 · M1 vs 축별 비음수 게이트 M2

Dunnhumby seed 42 validation에서 선택된 M2 구조를 바꾸지 않고 H&M 전체기간에 적용합니다. test와 holdout은 만들거나 평가하지 않습니다. 중간에 연결이 끊겨도 Drive의 epoch checkpoint에서 자동 재개됩니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '77b47e48cd07fd71cc8170b9cb7e37c057508acc'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA


In [ ]:
import json, torch
from IPython.display import display
from lightgcn_clv_axis_specific_gate_hm2y import (
    configure_axis_specific_gate_hm2y_run,
    preflight_summary,
    read_progress,
    run_experiment,
)

cfg = configure_axis_specific_gate_hm2y_run()
assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))


## 실행 전 또는 재연결 후 진행 상태 확인
아래 셀은 학습을 시작하지 않습니다. `completed`이면 실행 셀을 다시 누르지 말고 마지막 결과 셀만 확인하세요.

In [ ]:
print(json.dumps(read_progress(cfg.out_dir), ensure_ascii=False, indent=2))


## 학습 실행
기존 H&M 전체기간 M1 checkpoint가 설정까지 일치하면 재사용합니다. 없으면 M1을 먼저 한 번 학습합니다. M2는 매 epoch Drive에 저장되며 재실행 시 마지막 완료 epoch 다음부터 이어집니다.

In [ ]:
result_df = run_experiment(cfg)


In [ ]:
columns = [
    'selection_rule', 'model_id', 'role', 'selected_epoch',
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50', 'revenue@10', 'arp@10',
    'coverage@10', 'n_distinct@10', 'exposure_entropy@10',
    'eff_catalog@10', 'top10_share@10', 'top100_share@10',
    'value_alignment', 'gamma_n', 'gamma_v',
]
display(result_df[[column for column in columns if column in result_df.columns]])
print('최종 판정:')
print(json.dumps(result_df.attrs['decision'], ensure_ascii=False, indent=2))
print('결과 파일:', result_df.attrs['result_paths'])
